In [87]:
import polars as pl
_heter = pl.read_ndjson("/home/frank/files/programs/GraduationThesis/empirical/heter/异质性_sum3.jsonl",schema_overrides={'portfolio': pl.Utf8, 'date':pl.Date})


In [88]:
_heter

portfolio,date,value_id,bull_bear_label,equity,sum
str,date,i64,i64,i64,i64
"""600558""",2006-11-01,1,1,1,3
"""600281""",2006-04-01,1,1,1,3
"""600725""",2006-08-01,1,1,1,3
"""600403""",2006-12-01,1,1,1,3
"""000779""",2006-05-01,1,1,1,3
…,…,…,…,…,…
"""688119""",2024-09-01,1,1,1,3
"""000619""",2024-09-01,1,1,1,3
"""300557""",2024-10-01,1,1,1,3


In [89]:
_heter = _heter.with_columns(
    pl.col("portfolio")
      .map_elements(lambda x: [x])  # 每个元素包一层 list
      .alias("portfolio"),
    pl.col('date').dt.year().alias('year'),
    pl.col('date').dt.month().alias('month')
).select(pl.col(['year', 'month', 'portfolio', 'sum']))

In [90]:
df = pl.read_ndjson("/home/frank/files/programs/GraduationThesis/result/20260303_2125_baseline2_855b8d2a-c797-4b37-9953-6f007d737dca/a2_1/performance_and_reward_1.jsonl")

In [91]:
# 展开
df = df.with_columns(pl.col('data').struct.unnest()).drop('data')

# 处理权重
df = df.with_columns(
pl.col('decision_weights').list.get(0).alias('_weight'),
pl.col('decision_weights').list.get(1).alias('_risk_free_weight')
)

# 获取收益
df = df.with_columns((pl.col("performance").list.get(0) / (pl.col('_weight') + 1e-6)).alias("_rtr"))

In [92]:
df = df.join(_heter, on=['year','month','portfolio'], how='left')

In [93]:
df

year,month,portfolio,decision_weights,performance,normalized_performance,reward,_weight,_risk_free_weight,_rtr,sum
i64,i64,list[str],list[f64],list[f64],list[f64],f64,f64,f64,f64,i64
2019,12,"[""600996""]","[0.8127, 0.1873]","[0.013, -0.0, … 0.6958]","[0.013, -0.0, … 0.6958]",0.013,0.8127,0.1873,0.015996,null
2019,12,"[""603319""]","[0.3546, 0.6454]","[0.0101, -0.0, … 0.9381]","[0.0101, -0.0, … 0.9381]",0.0101,0.3546,0.6454,0.028483,null
2019,12,"[""002886""]","[0.6567, 0.3433]","[-0.0108, -0.0, … 0.9279]","[-0.0108, -0.0, … 0.9279]",-0.0108,0.6567,0.3433,-0.016446,null
2019,12,"[""603109""]","[1.0, 0.0]","[-0.1896, -0.0, … -0.0]","[-0.1896, -0.0, … -0.0]",-0.1896,1.0,0.0,-0.1896,null
2019,12,"[""603196""]","[0.398, 0.602]","[-0.0915, -0.0, … 0.9698]","[-0.0915, -0.0, … 0.9698]",-0.0915,0.398,0.602,-0.229899,null
…,…,…,…,…,…,…,…,…,…,…
2019,12,"[""002783""]","[1.0, 0.0]","[-0.0624, -0.0, … -0.0]","[-0.0624, -0.0, … -0.0]",-0.0624,1.0,0.0,-0.0624,null
2019,12,"[""603416""]","[0.2307, 0.7693]","[0.0065, -0.0, … 0.7793]","[0.0065, -0.0, … 0.7793]",0.0065,0.2307,0.7693,0.028175,null
2019,12,"[""603956""]","[0.8774, 0.1226]","[-0.0458, -0.0, … 0.5367]","[-0.0458, -0.0, … 0.5367]",-0.0458,0.8774,0.1226,-0.0522,null


In [94]:
heter_df = df.filter(pl.col('sum') == 3)
not_heter_df = df.filter((pl.col('sum').is_null()) | (pl.col('sum') != 3))
pos_df = not_heter_df.filter(pl.col('reward') >= 0)
neg_df = not_heter_df.filter(pl.col('reward') < 0)

In [95]:
# 处理异质性
class SELF:
    def __init__(self, short_limit, mix_weight):
        self._short_limit = short_limit
        self._mix_weight = mix_weight
self = SELF(-0.05, 0.5)
mix_weight = 0.5
import random
if not heter_df.is_empty():
    # 制造一个相反的决策
    # 当收益为[-0.03,0.03]时，决策为[1-short_limt, short_limit]
    kh = (self._short_limit - (1-self._short_limit)) / (0.03 - (-0.03))
    heter_df = heter_df.with_columns(
        (kh * (pl.col('_rtr') - (-0.03)) + (1-self._short_limit)).alias('_fix_weight')
    )
    # 计算异质性权重
    heter_df = heter_df.with_columns(
        (pl.col('_weight') * (1-mix_weight) + mix_weight * pl.col('_fix_weight')).alias('_weight')
    )
    # 奖励衰减
    heter_df = heter_df.with_columns(pl.col('reward') - random.uniform(0.01, 0.05))           
    # 去掉_fix_weight
    heter_df = heter_df.drop('_fix_weight')

# 处理负收益
if not neg_df.is_empty():
    # 制造一个同向的决策
    # 当收益为[-0.01,0.03]时，决策为[short_limit,1.0-short_limit]
    kn = (1-self._short_limit - self._short_limit) / (0.03 - (-0.01))
    neg_df = neg_df.with_columns(
        (kn * (pl.col('_rtr') - (-0.01)) + self._short_limit).alias('_fix_weight')
    )
    # 计算负收益权重
    neg_df = neg_df.with_columns(
        (pl.col('_weight') * (1-mix_weight) + mix_weight * pl.col('_fix_weight')).alias('_weight')
    )
    # 奖励增加
    neg_df = neg_df.with_columns(pl.col('reward') + random.uniform(0.01, 0.05))
    # 去掉_fix_weight
    neg_df = neg_df.drop('_fix_weight')

# 合并
df_list = [heter_df, pos_df, neg_df]
df_list = [df for df in df_list if not df.is_empty()]
if df_list:
    df = pl.concat(df_list)

# 将decision_weights截断
df = df.with_columns(pl.col('_weight').clip(self._short_limit, 1-self._short_limit).alias('_weight'))
df = df.with_columns((1 - pl.col('_weight')).alias('_risk_free_weight'))

In [100]:
# 恢复decision_weights结构
df = df.with_columns(pl.concat_list(pl.col('_weight'), pl.col('_risk_free_weight')).alias('decision_weights'))
df = df.drop('_weight', '_risk_free_weight', '_rtr','sum')

In [104]:
df = df.with_columns(pl.struct(pl.col('decision_weights'), pl.col('performance'), pl.col('normalized_performance'), pl.col('reward')).alias('data'))

In [106]:
df = df.drop('decision_weights', 'performance', 'normalized_performance', 'reward')

In [108]:
df.to_dicts()

[{'year': 2019,
  'month': 12,
  'portfolio': ['600996'],
  'data': {'decision_weights': [0.8127, 0.1873],
   'performance': [0.013, -0.0, 0.0, -0.0, 0.6958],
   'normalized_performance': [0.013, -0.0, 0.0, -0.0, 0.6958],
   'reward': 0.013}},
 {'year': 2019,
  'month': 12,
  'portfolio': ['603319'],
  'data': {'decision_weights': [0.3546, 0.6454],
   'performance': [0.0101, -0.0, 0.0, -0.0, 0.9381],
   'normalized_performance': [0.0101, -0.0, 0.0, -0.0, 0.9381],
   'reward': 0.0101}},
 {'year': 2019,
  'month': 12,
  'portfolio': ['603279'],
  'data': {'decision_weights': [1.0, 0.0],
   'performance': [0.1233, -0.0, 0.0, -0.0, -0.0],
   'normalized_performance': [0.1233, -0.0, 0.0, -0.0, -0.0],
   'reward': 0.1233}},
 {'year': 2019,
  'month': 12,
  'portfolio': ['603722'],
  'data': {'decision_weights': [0.5, 0.5],
   'performance': [0.0441, -0.0, 0.0, -0.0, 1.0],
   'normalized_performance': [0.0441, -0.0, 0.0, -0.0, 1.0],
   'reward': 0.0441}},
 {'year': 2019,
  'month': 12,
  'por